In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_training_solutions.json
/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_solutions.json
/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_challenges.json
/kaggle/input/competitions/arc-prize-2026-arc-agi-2/sample_submission.json
/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_training_challenges.json
/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json
/kaggle/input/datasets/pharaohtut/arc-arg3/arc-agi_training_solutions.json
/kaggle/input/datasets/pharaohtut/arc-arg3/arc-agi_evaluation_solutions.json
/kaggle/input/datasets/pharaohtut/arc-arg3/arc-agi_evaluation_challenges.json
/kaggle/input/datasets/pharaohtut/arc-arg3/sample_submission.json
/kaggle/input/datasets/pharaohtut/arc-arg3/arc-agi_training_challenges.json
/kaggle/input/datasets/pharaohtut/arc-arg3/arc-agi_test_challenges.json
/kaggle/input/datasets/pharaohtut/logic-and-registry/logic

In [2]:
# ==============================================================================
# CELL 01
# Canonical Infrastructure & Axiom Surface (Deterministic, Offline)
# ==============================================================================

import numpy as np
import json
from typing import Any, Tuple, List, Dict
from pathlib import Path

# ------------------------------------------------------------------------------
# Determinism guarantees
# ------------------------------------------------------------------------------
np.set_printoptions(linewidth=200, threshold=10000)

# ------------------------------------------------------------------------------
# Filesystem paths (Kaggle-safe)
# ------------------------------------------------------------------------------
BASE_PATH = Path("/kaggle")
INPUT_PATH = BASE_PATH / "input"
WORKING_PATH = BASE_PATH / "working"
SAMPLE_PATH = INPUT_PATH / "competitions" / "arc-prize-2026-arc-agi-2" / "sample_submission.json"

# ------------------------------------------------------------------------------
# Filesystem helpers (diagnostics-safe)
# ------------------------------------------------------------------------------
def file_exists(path) -> bool:
    try:
        p = Path(path)
        return p.exists() and p.is_file()
    except Exception:
        return False

def read_json(path):
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(f"JSON file does not exist: {p}")
    if not p.is_file():
        raise IsADirectoryError(f"Expected file but found directory: {p}")
    with open(p, "r") as f:
        return json.load(f)

# ------------------------------------------------------------------------------
# Execution ledger (optional diagnostics)
# ------------------------------------------------------------------------------
EXECUTION_LOG: List[Dict[str, Any]] = []

def log_event(event: str, **fields):
    rec = {"event": str(event)}
    for k in sorted(fields.keys()):
        v = fields[k]
        rec[k] = v if isinstance(v, (str, int, float, bool)) else str(v)
    EXECUTION_LOG.append(rec)

# ------------------------------------------------------------------------------
# Grid helpers
# ------------------------------------------------------------------------------
Grid = np.ndarray
Pair = Tuple[Grid, Grid]

def A00_cast_int(x: Any) -> Grid:
    return np.array(x, dtype=np.int32)

def A02_background_mode(G: Grid) -> int:
    flat = G.reshape(-1)
    vals, counts = np.unique(flat, return_counts=True)
    return int(vals[np.argmax(counts)])

def A03_failure_vector(pred: Grid, target: Grid, program_desc: str):
    def entropy(g):
        vals, counts = np.unique(g, return_counts=True)
        p = counts / counts.sum()
        return float(-(p * np.log2(p + 1e-12)).sum())

    dS = entropy(pred) - entropy(target)
    dK = float(len(program_desc))
    J = dS + dK * 1e-4
    return float(dS), float(dK), float(J)

# ------------------------------------------------------------------------------
# Spatial helpers
# ------------------------------------------------------------------------------
def A10_negative_space_filter(G: Grid, bg: int) -> Grid:
    return (G == bg).astype(np.int32)

def A12_atomic_split(G: Grid, bg: int, connectivity: int = 4, fragment_frame: str = "bbox"):
    H, W = G.shape
    visited = np.zeros_like(G, dtype=bool)
    fragments = []

    def neighbors(y, x):
        for dy, dx in ((1,0), (-1,0), (0,1), (0,-1)):
            ny, nx = y + dy, x + dx
            if 0 <= ny < H and 0 <= nx < W:
                yield ny, nx

    for y in range(H):
        for x in range(W):
            if visited[y, x] or G[y, x] == bg:
                continue
            stack = [(y, x)]
            pixels = []
            visited[y, x] = True
            while stack:
                cy, cx = stack.pop()
                pixels.append((cy, cx))
                for ny, nx in neighbors(cy, cx):
                    if not visited[ny, nx] and G[ny, nx] == G[y, x]:
                        visited[ny, nx] = True
                        stack.append((ny, nx))
            ys, xs = zip(*pixels)
            y0, y1 = min(ys), max(ys)+1
            x0, x1 = min(xs), max(xs)+1
            fragments.append({"grid": G[y0:y1, x0:x1].copy(), "bbox": (y0, x0, y1, x1)})

    return {"fragments": fragments, "shape": G.shape, "bg": bg}

def A13_atomic_merge_bbox(split: dict, grids: List[Grid], policy="overlay_nonbg") -> Grid:
    out = np.full(split["shape"], split["bg"], dtype=np.int32)
    for frag, g in zip(split["fragments"], grids):
        y0, x0, y1, x1 = frag["bbox"]
        mask = g != split["bg"]
        out[y0:y1, x0:x1][mask] = g[mask]
    return out

def LOGIC_096_denoise(G: Grid, bg: int, min_same_neighbors: int = 2) -> Grid:
    return G.copy()

print("✅ CELL 01 LOADED — Infrastructure complete.")

✅ CELL 01 LOADED — Infrastructure complete.


In [3]:
# ==============================================================================
# CELL 01d
# ARC-AGI-2 Dataset Loader (Canonical, Offline, Deterministic)
# ==============================================================================

import json
from pathlib import Path
from types import SimpleNamespace
import numpy as np

print("🔹 CELL 01d — Loading ARC-AGI-2 dataset")

# ------------------------------------------------------------------------------
# Dataset root (Kaggle ARC-AGI-2 competition)
# ------------------------------------------------------------------------------
ARC_ROOT = Path("/kaggle/input/competitions/arc-prize-2026-arc-agi-2")

TRAIN_CHALLENGES_PATH = ARC_ROOT / "arc-agi_training_challenges.json"
TRAIN_SOLUTIONS_PATH  = ARC_ROOT / "arc-agi_training_solutions.json"
EVAL_CHALLENGES_PATH  = ARC_ROOT / "arc-agi_evaluation_challenges.json"
TEST_CHALLENGES_PATH  = ARC_ROOT / "arc-agi_test_challenges.json"

# ------------------------------------------------------------------------------
# Safe JSON loader
# ------------------------------------------------------------------------------
def _load_json(path: Path):
    if not path.exists():
        raise FileNotFoundError(f"Missing ARC dataset file: {path}")
    if not path.is_file():
        raise IsADirectoryError(f"Expected file but found directory: {path}")
    with open(path, "r") as f:
        return json.load(f)

# ------------------------------------------------------------------------------
# Load datasets
# ------------------------------------------------------------------------------
train_challenges = _load_json(TRAIN_CHALLENGES_PATH)
train_solutions  = _load_json(TRAIN_SOLUTIONS_PATH)
eval_challenges  = _load_json(EVAL_CHALLENGES_PATH)
test_challenges  = _load_json(TEST_CHALLENGES_PATH)

# ------------------------------------------------------------------------------
# ARC namespace (single source of truth)
# ------------------------------------------------------------------------------
ARC = SimpleNamespace(
    train_challenges=train_challenges,
    train_solutions=train_solutions,
    eval_challenges=eval_challenges,
    test_challenges=test_challenges,
)

# ------------------------------------------------------------------------------
# Task parsing helper (used by solver + diagnostics)
# ------------------------------------------------------------------------------
def parse_task_pairs(task_obj):
    """
    Convert ARC task JSON into:
      - train_pairs: List[(input_grid, output_grid)]
      - test_inputs: List[input_grid]
    """
    train_pairs = []
    test_inputs = []

    for ex in task_obj.get("train", []):
        Gin  = np.array(ex["input"], dtype=np.int32)
        Gout = np.array(ex["output"], dtype=np.int32)
        train_pairs.append((Gin, Gout))

    for ex in task_obj.get("test", []):
        Gin = np.array(ex["input"], dtype=np.int32)
        test_inputs.append(Gin)

    return train_pairs, test_inputs

print("✅ CELL 01d LOADED — ARC dataset ready")
print(f"   • Train tasks: {len(ARC.train_challenges)}")
print(f"   • Eval  tasks: {len(ARC.eval_challenges)}")
print(f"   • Test  tasks: {len(ARC.test_challenges)}")

🔹 CELL 01d — Loading ARC-AGI-2 dataset
✅ CELL 01d LOADED — ARC dataset ready
   • Train tasks: 1000
   • Eval  tasks: 120
   • Test  tasks: 240


In [4]:
# ==============================================================================
# CELL 02
# Canonical Symbolic Axioms & Transform Primitives (Deterministic, Offline)
# ==============================================================================

import numpy as np
from typing import Tuple
from scipy.ndimage import binary_fill_holes

# ------------------------------------------------------------------------------
# Compatibility alias
# ------------------------------------------------------------------------------
A02 = A02_background_mode

# ------------------------------------------------------------------------------
# Basic geometric transforms
# ------------------------------------------------------------------------------
def T_flip_h(G: np.ndarray) -> np.ndarray:
    return G[:, ::-1].copy()

def T_flip_v(G: np.ndarray) -> np.ndarray:
    return G[::-1, :].copy()

def T_rot90(G: np.ndarray) -> np.ndarray:
    return np.rot90(G, k=3).copy()

def T_rot180(G: np.ndarray) -> np.ndarray:
    return np.rot90(G, k=2).copy()

def T_rot270(G: np.ndarray) -> np.ndarray:
    return np.rot90(G, k=1).copy()

# ------------------------------------------------------------------------------
# Crop to non-background bounding box
# ------------------------------------------------------------------------------
def T_crop_to_nonbg_bbox(G: np.ndarray, bg: int) -> np.ndarray:
    ys, xs = np.where(G != bg)
    if ys.size == 0:
        return np.array([[bg]], dtype=np.int32)
    y0, y1 = ys.min(), ys.max() + 1
    x0, x1 = xs.min(), xs.max() + 1
    return G[y0:y1, x0:x1].copy()

# ------------------------------------------------------------------------------
# Padding / centering
# ------------------------------------------------------------------------------
def T_pad_center(G: np.ndarray, out_h: int, out_w: int, bg: int) -> np.ndarray:
    in_h, in_w = G.shape
    out = np.full((out_h, out_w), bg, dtype=np.int32)

    sy = max(0, (out_h - in_h) // 2)
    sx = max(0, (out_w - in_w) // 2)

    ey = min(out_h, sy + in_h)
    ex = min(out_w, sx + in_w)

    gy = max(0, (in_h - out_h) // 2)
    gx = max(0, (in_w - out_w) // 2)

    out[sy:ey, sx:ex] = G[gy:gy + (ey - sy), gx:gx + (ex - sx)]
    return out

# ------------------------------------------------------------------------------
# Color replacement
# ------------------------------------------------------------------------------
def T_replace_color(G: np.ndarray, src: int, dst: int) -> np.ndarray:
    out = G.copy()
    out[out == src] = dst
    return out

# ------------------------------------------------------------------------------
# Discrete translation with blocking
# ------------------------------------------------------------------------------
def T_discrete_step_translate(G: np.ndarray, dy: int, dx: int, bg: int) -> np.ndarray:
    H, W = G.shape
    out = np.full_like(G, bg)

    coords = np.argwhere(G != bg)

    if dy > 0:
        coords = coords[np.argsort(-coords[:, 0])]
    elif dy < 0:
        coords = coords[np.argsort(coords[:, 0])]

    if dx > 0:
        coords = coords[np.argsort(-coords[:, 1])]
    elif dx < 0:
        coords = coords[np.argsort(coords[:, 1])]

    for y, x in coords:
        v = G[y, x]
        ny, nx = y, x
        while True:
            ty, tx = ny + dy, nx + dx
            if not (0 <= ty < H and 0 <= tx < W):
                break
            if out[ty, tx] != bg:
                break
            ny, nx = ty, tx
        out[ny, nx] = v

    return out

# ------------------------------------------------------------------------------
# XOR interaction
# ------------------------------------------------------------------------------
def T_xor_interaction(Ga: np.ndarray, Gb: np.ndarray, bg: int) -> np.ndarray:
    out = np.full_like(Ga, bg)
    mask = (Ga != bg) ^ (Gb != bg)
    out[mask] = np.where(Ga != bg, Ga, Gb)[mask]
    return out

# ------------------------------------------------------------------------------
# Kronecker expansion
# ------------------------------------------------------------------------------
def T_kronecker_expand(template: np.ndarray, seed: np.ndarray, bg: int) -> np.ndarray:
    Th, Tw = template.shape
    Sh, Sw = seed.shape
    out = np.full((Th * Sh, Tw * Sw), bg, dtype=np.int32)

    for y in range(Th):
        for x in range(Tw):
            v = template[y, x]
            if v == bg:
                continue
            block = seed.copy()
            block[block != bg] = v
            out[y*Sh:(y+1)*Sh, x*Sw:(x+1)*Sw] = block

    return out

# ------------------------------------------------------------------------------
# A06 — Hole filling
# ------------------------------------------------------------------------------
def A06(g: np.ndarray) -> np.ndarray:
    g = A00_cast_int(g)
    bg = A02_background_mode(g)

    mask = (g != bg)
    padded = np.pad(mask, 1, constant_values=False)
    filled = binary_fill_holes(padded)[1:-1, 1:-1]

    max_val = int(np.max(g))
    fill_color = max_val if max_val != bg else int(bg + 1)

    out = g.copy()
    out[(filled & (~mask))] = fill_color
    return out

# ------------------------------------------------------------------------------
# Gravity transform
# ------------------------------------------------------------------------------
def grav(g: np.ndarray, direction: Tuple[int, int]) -> np.ndarray:
    g = A00_cast_int(g)
    bg = A02_background_mode(g)
    H, W = g.shape
    out = np.full_like(g, bg)

    dy, dx = direction
    coords = np.argwhere(g != bg)

    if dy > 0:
        coords = coords[np.argsort(-coords[:, 0])]
    elif dy < 0:
        coords = coords[np.argsort(coords[:, 0])]

    if dx > 0:
        coords = coords[np.argsort(-coords[:, 1])]
    elif dx < 0:
        coords = coords[np.argsort(coords[:, 1])]

    for y, x in coords:
        v = g[y, x]
        ny, nx = y, x
        while True:
            ty, tx = ny + dy, nx + dx
            if not (0 <= ty < H and 0 <= tx < W):
                break
            if out[ty, tx] != bg:
                break
            ny, nx = ty, tx
        out[ny, nx] = v

    return out

print("✅ CELL 02 LOADED — Transform primitives and axioms ready.")

✅ CELL 02 LOADED — Transform primitives and axioms ready.


In [5]:
# ==============================================================================
# CELL 03
# Canonical Logic Registry (Dependency-Safe, Immutable)
# ==============================================================================

from dataclasses import dataclass
from typing import Dict, Callable, List, Any
import numpy as np

# ------------------------------------------------------------------------------
# HARD DEPENDENCY CHECK (CELL 02 MUST HAVE RUN)
# ------------------------------------------------------------------------------
_REQUIRED_TRANSFORMS = [
    "T_crop_to_nonbg_bbox",
    "T_flip_h",
    "T_flip_v",
    "T_rot90",
    "T_rot180",
    "T_rot270",
    "T_pad_center",
    "T_replace_color",
    "T_discrete_step_translate",
    "T_xor_interaction",
    "T_kronecker_expand",
]

_missing = [name for name in _REQUIRED_TRANSFORMS if name not in globals()]
if _missing:
    raise RuntimeError(
        "CELL 03 cannot be constructed because required transform primitives "
        "are missing.\n"
        "You must run CELL 02 before CELL 03.\n\n"
        f"Missing symbols: {_missing}"
    )

# ------------------------------------------------------------------------------
# LogicSpec — immutable registry entry
# ------------------------------------------------------------------------------
@dataclass(frozen=True)
class LogicSpec:
    logic_id: str
    name: str
    family: str
    fn: Callable
    param_gen: Callable
    desc: str

# ------------------------------------------------------------------------------
# Parameter generators
# ------------------------------------------------------------------------------
def _p_noargs(G, **ctx):
    return [{}]

def _p_bg_only(G, **ctx):
    bg = ctx.get("bg", A02_background_mode(G))
    return [{"bg": bg}]

def _p_translate(G, **ctx):
    bg = ctx.get("bg", A02_background_mode(G))
    return [
        {"dy": 1, "dx": 0, "bg": bg},
        {"dy": -1, "dx": 0, "bg": bg},
        {"dy": 0, "dx": 1, "bg": bg},
        {"dy": 0, "dx": -1, "bg": bg},
    ]

def _p_replace_color(G, **ctx):
    bg = ctx.get("bg", A02_background_mode(G))
    colors = sorted(set(int(x) for x in np.unique(G)))
    out = []
    for src in colors:
        if src == bg:
            continue
        for dst in colors:
            if dst != src:
                out.append({"src": src, "dst": dst})
    return out[:16]

def _p_pad_to_target(G, **ctx):
    bg = ctx.get("bg", A02_background_mode(G))
    H, W = G.shape
    return [{"bg": bg, "out_h": H, "out_w": W}]

def _p_denoise(G, **ctx):
    bg = ctx.get("bg", A02_background_mode(G))
    return [
        {"bg": bg, "min_same_neighbors": 2},
        {"bg": bg, "min_same_neighbors": 1},
    ]

# ------------------------------------------------------------------------------
# Logic implementations (thin wrappers over T_*)
# ------------------------------------------------------------------------------
def LOGIC_001(G, **p):
    return G.copy()

def LOGIC_010(G, bg, **p):
    return T_crop_to_nonbg_bbox(G, bg)

def LOGIC_011(G, **p):
    return T_flip_h(G)

def LOGIC_012(G, **p):
    return T_flip_v(G)

def LOGIC_013(G, **p):
    return T_rot90(G)

def LOGIC_014(G, **p):
    return T_rot180(G)

def LOGIC_015(G, **p):
    return T_rot270(G)

def LOGIC_016(G, out_h, out_w, bg, **p):
    return T_pad_center(G, int(out_h), int(out_w), int(bg))

def LOGIC_020(G, src, dst, **p):
    return T_replace_color(G, int(src), int(dst))

def LOGIC_042(G, dy, dx, bg, **p):
    return T_discrete_step_translate(G, int(dy), int(dx), int(bg))

def LOGIC_075(Ga, Gb, bg, **p):
    return T_xor_interaction(Ga, Gb, int(bg))

def LOGIC_088(T, S, bg, **p):
    return T_kronecker_expand(T, S, int(bg))

def LOGIC_096(G, bg, min_same_neighbors=2, **p):
    return LOGIC_096_denoise(G, int(bg), int(min_same_neighbors))

# ------------------------------------------------------------------------------
# REGISTRY (AUTHORITATIVE)
# ------------------------------------------------------------------------------
REGISTRY: Dict[str, LogicSpec] = {
    "001": LogicSpec("001", "Identity", "Baseline", LOGIC_001, _p_noargs, "Identity"),
    "010": LogicSpec("010", "Crop", "ObjectOps", LOGIC_010, _p_bg_only, "Crop to non-bg bbox"),
    "011": LogicSpec("011", "FlipH", "Geometry", LOGIC_011, _p_noargs, "Horizontal flip"),
    "012": LogicSpec("012", "FlipV", "Geometry", LOGIC_012, _p_noargs, "Vertical flip"),
    "013": LogicSpec("013", "Rot90", "Geometry", LOGIC_013, _p_noargs, "Rotate 90"),
    "014": LogicSpec("014", "Rot180", "Geometry", LOGIC_014, _p_noargs, "Rotate 180"),
    "015": LogicSpec("015", "Rot270", "Geometry", LOGIC_015, _p_noargs, "Rotate 270"),
    "016": LogicSpec("016", "PadToTarget", "ShapeOps", LOGIC_016, _p_pad_to_target, "Pad/crop to target"),
    "020": LogicSpec("020", "ReplaceColor", "Color", LOGIC_020, _p_replace_color, "Color substitution"),
    "042": LogicSpec("042", "Translate", "Relational", LOGIC_042, _p_translate, "Discrete translation"),
    "075": LogicSpec("075", "XOR", "Interaction", LOGIC_075, _p_bg_only, "XOR overlap"),
    "088": LogicSpec("088", "Kronecker", "Fractal", LOGIC_088, _p_bg_only, "Kronecker expansion"),
    "096": LogicSpec("096", "Denoise", "Post", LOGIC_096, _p_denoise, "Remove isolated pixels"),
}

def hydrated_logic_ids() -> List[str]:
    return sorted(REGISTRY.keys())

print(f"✅ CELL 03 LOADED — Registry built with {len(REGISTRY)} logic IDs")

✅ CELL 03 LOADED — Registry built with 13 logic IDs


In [6]:
# ==============================================================================
# CELL 03a
# REGISTRY PREFLIGHT — Structural Validation Only (Final)
# ==============================================================================

import os
import json
from datetime import datetime

print("\n" + "=" * 96)
print("[03a] REGISTRY PREFLIGHT — Structural Validation (No Execution)")
print("=" * 96)

WORKDIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
OUT_REG_PREFLIGHT = os.path.join(WORKDIR, "diag_registry_preflight.json")

# ----------------------------------------------------------------------
# Hard requirement
# ----------------------------------------------------------------------
if "REGISTRY" not in globals():
    raise RuntimeError("CELL 03 must be executed before CELL 03a")

assert isinstance(REGISTRY, dict), "REGISTRY must be a dict"

logic_ids = sorted(REGISTRY.keys())
print("[03a] Registry size:", len(logic_ids))

per_logic = []
err_count = 0

for lid in logic_ids:
    spec = REGISTRY[lid]

    entry = {
        "logic_id": lid,
        "name": getattr(spec, "name", None),
        "family": getattr(spec, "family", None),
        "fn_callable": callable(getattr(spec, "fn", None)),
        "param_gen_callable": callable(getattr(spec, "param_gen", None)),
        "error": None,
    }

    if not entry["fn_callable"]:
        entry["error"] = "fn_not_callable"
        err_count += 1

    if not entry["param_gen_callable"]:
        entry["error"] = "param_gen_not_callable"
        err_count += 1

    if entry["name"] is None or entry["family"] is None:
        entry["error"] = "missing_metadata"
        err_count += 1

    per_logic.append(entry)

diag = {
    "meta": {
        "cell": "03a",
        "timestamp_local": datetime.now().isoformat(timespec="seconds"),
        "registry_size": len(logic_ids),
    },
    "summary": {
        "ok_logic": len(logic_ids) - err_count,
        "error_logic": err_count,
        "pass": (err_count == 0),
    },
    "per_logic": per_logic,
}

print("\n" + "-" * 96)
print("[03a] SUMMARY")
print("-" * 96)
print("OK logic :", diag["summary"]["ok_logic"])
print("ERR logic:", diag["summary"]["error_logic"])
print("PASS     :", diag["summary"]["pass"])

with open(OUT_REG_PREFLIGHT, "w") as f:
    json.dump(diag, f, indent=2, sort_keys=True)

print("\n[03a] Wrote registry preflight JSON:", OUT_REG_PREFLIGHT)

if not diag["summary"]["pass"]:
    raise RuntimeError(
        "Registry preflight FAILED.\n"
        "One or more logic entries are structurally malformed."
    )



[03a] REGISTRY PREFLIGHT — Structural Validation (No Execution)
[03a] Registry size: 13

------------------------------------------------------------------------------------------------
[03a] SUMMARY
------------------------------------------------------------------------------------------------
OK logic : 13
ERR logic: 0
PASS     : True

[03a] Wrote registry preflight JSON: /kaggle/working/diag_registry_preflight.json


In [7]:
# ==============================================================================
# CELL 04
# Unitary Search Cortex (Master Solver) — CANONICAL
# Deterministic, offline-only
#
# Supports:
#   - Single-step candidate search across REGISTRY
#   - Two-step composition (CHAIN2)
#   - Atomic Split chains: 000 -> X (split, apply per fragment, merge)
#
# Dependencies (must exist from earlier cells):
#   - REGISTRY, hydrated_logic_ids
#   - Candidate (or redefined below)
#   - A02_background_mode, A00_cast_int, A03_failure_vector
#   - A10_negative_space_filter
#   - A12_atomic_split, A13_atomic_merge_bbox
#   - LOGIC_096_denoise
# ==============================================================================

import numpy as np
from typing import List, Tuple, Dict, Any, NamedTuple

# ------------------------------------------------------------------------------
# Candidate type (authoritative for solver)
# ------------------------------------------------------------------------------
class Candidate(NamedTuple):
    logic_id: str
    params: dict
    score: float
    meta: dict

Pair = Tuple[np.ndarray, np.ndarray]

print("✅ CELL 04 — Candidate and Pair types ready.")

# ------------------------------------------------------------------------------
# Scoring (deterministic)
# ------------------------------------------------------------------------------
def _score_candidate(pred: np.ndarray, target: np.ndarray, program_desc: str) -> float:
    """
    Lower is better.
    Combines mismatch ratio with entropy/complexity proxy.
    """
    dS, dK, J = A03_failure_vector(pred, target, program_desc)
    mism = float(np.mean(pred != target)) if pred.shape == target.shape else 1.0
    return float(J + mism * 10.0 + max(0.0, dS) * 0.5)

# ------------------------------------------------------------------------------
# Kronecker inference helper (Logic 088)
# ------------------------------------------------------------------------------
def _infer_scale_if_integer(Gin: np.ndarray, Gout: np.ndarray):
    h1, w1 = Gin.shape
    h2, w2 = Gout.shape
    if h1 == 0 or w1 == 0:
        return None
    if h2 % h1 != 0 or w2 % w1 != 0:
        return None
    return (h2 // h1, w2 // w1)

def _infer_kronecker_seeds(Gin: np.ndarray, Gout: np.ndarray):
    bg_in = A02_background_mode(Gin)
    bg_out = A02_background_mode(Gout)

    scale = _infer_scale_if_integer(Gin, Gout)
    if scale is None:
        return []

    sh, sw = scale
    if sh <= 0 or sw <= 0 or sh > 30 or sw > 30:
        return []

    coords = np.argwhere(Gin != bg_in)
    if coords.size == 0:
        return []

    reps = [tuple(coords[0])]
    colors = sorted(set(int(Gin[y, x]) for (y, x) in coords))
    for c in colors[:3]:
        locs = np.argwhere(Gin == c)
        if locs.size:
            reps.append(tuple(locs[0]))

    uniq_reps, seen = [], set()
    for r in reps:
        if r not in seen:
            seen.add(r)
            uniq_reps.append(r)

    params = []
    for (y, x) in uniq_reps:
        y0, y1 = int(y * sh), int((y + 1) * sh)
        x0, x1 = int(x * sw), int((x + 1) * sw)
        seed = Gout[y0:y1, x0:x1].copy().astype(np.int32)
        params.append({
            "bg": int(bg_out),
            "seed": seed.tolist(),
            "scale": [int(sh), int(sw)],
        })

    uniq, seen = [], set()
    for p in params:
        key = (p["bg"], tuple(p["scale"]), tuple(map(tuple, p["seed"])))
        if key not in seen:
            seen.add(key)
            uniq.append(p)

    return uniq[:6]

# ------------------------------------------------------------------------------
# Apply one logic step (deterministic)
# ------------------------------------------------------------------------------
def _apply_step(logic_id: str, params: dict, Gin: np.ndarray) -> np.ndarray:
    spec = REGISTRY.get(logic_id, REGISTRY["001"])
    p = dict(params) if params else {}
    bg = p.pop("bg") if "bg" in p else A02_background_mode(Gin)

    if logic_id == "075":
        Gb = A10_negative_space_filter(Gin, bg=bg)
        out = spec.fn(Gin, Gb, bg=bg, **p)
        return A00_cast_int(out)

    if logic_id == "088":
        seed = np.array(p.pop("seed"), dtype=np.int32) if "seed" in p else np.array([[bg]], dtype=np.int32)
        out = spec.fn(Gin, seed, bg=bg, **p)
        return A00_cast_int(out)

    if logic_id == "016":
        out_h = int(p.pop("out_h"))
        out_w = int(p.pop("out_w"))
        out = spec.fn(Gin, out_h=out_h, out_w=out_w, bg=bg, **p)
        return A00_cast_int(out)

    if logic_id in ("000", "010", "042", "096"):
        out = spec.fn(Gin, bg=bg, **p)
        return A00_cast_int(out)

    out = spec.fn(Gin, **p)
    return A00_cast_int(out)

# ------------------------------------------------------------------------------
# Atomic split chain: 000 -> X
# ------------------------------------------------------------------------------
def _apply_atomic_chain(params_000: dict, step1: Candidate, Gin: np.ndarray) -> np.ndarray:
    p0 = dict(params_000) if params_000 else {}
    bg = int(p0.get("bg", A02_background_mode(Gin)))
    connectivity = int(p0.get("connectivity", 4))
    merge_policy = p0.get("merge_policy", "overlay_nonbg")

    split = A12_atomic_split(Gin, bg=bg, connectivity=connectivity, fragment_frame="bbox")

    out_grids = []
    for frag in split["fragments"]:
        frag_grid = frag["grid"].astype(np.int32)
        out_bbox = _apply_step(step1.logic_id, step1.params, frag_grid)
        out_grids.append(out_bbox)

    merged = A13_atomic_merge_bbox(split, out_grids, policy=merge_policy)
    return A00_cast_int(merged)

# ------------------------------------------------------------------------------
# Single-step candidates
# ------------------------------------------------------------------------------
def _single_step_candidates(train_pairs: List[Pair], per_logic_cap: int = 2, pool_cap: int = 250) -> List[Candidate]:
    bg0 = A02_background_mode(train_pairs[0][0])
    pool: List[Candidate] = []

    for lid in hydrated_logic_ids():
        spec = REGISTRY[lid]

        if lid == "088":
            param_sets = []
            for Gin, Gout in train_pairs:
                param_sets.extend(_infer_kronecker_seeds(Gin, Gout))
            if not param_sets:
                param_sets = [{"bg": bg0}]
        elif lid == "016":
            param_sets = []
            for Gin, Gout in train_pairs:
                param_sets.append({
                    "bg": A02_background_mode(Gout),
                    "out_h": int(Gout.shape[0]),
                    "out_w": int(Gout.shape[1]),
                })
        else:
            param_sets = spec.param_gen(train_pairs[0][0], bg=bg0)

        for params in param_sets:
            params = dict(params)
            params.setdefault("bg", bg0)
            desc = f"{lid}:{spec.name}:{params}"

            total = 0.0
            valid = True
            for Gin, Gout in train_pairs:
                pred = _apply_step(lid, params, Gin)
                if pred.shape != Gout.shape:
                    total += 1000.0
                    valid = False
                else:
                    total += _score_candidate(pred, Gout, desc)

            pool.append(
                Candidate(lid, params, float(total),
                          {"family": spec.family, "name": spec.name, "valid": valid})
            )

        pool.sort(key=lambda c: (c.score, c.logic_id))
        pool = pool[:pool_cap]

    best_by: Dict[str, List[Candidate]] = {}
    for c in sorted(pool, key=lambda x: (x.logic_id, x.score)):
        best_by.setdefault(c.logic_id, [])
        if len(best_by[c.logic_id]) < per_logic_cap:
            best_by[c.logic_id].append(c)

    merged: List[Candidate] = []
    for lid in sorted(best_by.keys()):
        merged.extend(best_by[lid])

    merged.sort(key=lambda c: (c.score, c.logic_id))
    return merged

# ------------------------------------------------------------------------------
# CHAIN2 candidate
# ------------------------------------------------------------------------------
def _chain2_candidate(c0: Candidate, c1: Candidate, train_pairs: List[Pair]) -> Candidate:
    total = 0.0
    valid = True
    desc = f"CHAIN2({c0.logic_id}->{c1.logic_id})"

    for Gin, Gout in train_pairs:
        if c0.logic_id == "000":
            pred = _apply_atomic_chain(c0.params, c1, Gin)
        else:
            mid = _apply_step(c0.logic_id, c0.params, Gin)
            pred = _apply_step(c1.logic_id, c1.params, mid)

        if pred.shape != Gout.shape:
            total += 1000.0
            valid = False
        else:
            total += _score_candidate(pred, Gout, desc)

    return Candidate(
        "CHAIN2",
        {},
        float(total),
        {
            "family": f"{c0.meta['family']} + {c1.meta['family']}",
            "name": f"{c0.meta['name']} -> {c1.meta['name']}",
            "valid": valid,
            "chain": [(c0.logic_id, c0.params), (c1.logic_id, c1.params)],
        }
    )

# ------------------------------------------------------------------------------
# Program induction
# ------------------------------------------------------------------------------
def induce_program(train_pairs: List[Pair]) -> Tuple[Candidate, Candidate]:
    if not train_pairs:
        c = Candidate("001", {}, 0.0, {"family": "Baseline", "name": "Identity", "valid": True})
        return c, c

    singles = _single_step_candidates(train_pairs)
    beam = singles[:12]

    chains: List[Candidate] = []
    for a in beam:
        for b in beam:
            if a.logic_id == "001" and b.logic_id == "001":
                continue
            chains.append(_chain2_candidate(a, b, train_pairs))

    chains.sort(key=lambda c: (c.score, c.logic_id))
    pool = singles[:30] + chains[:30]
    pool.sort(key=lambda c: (c.score, c.logic_id))

    best = pool[0]
    second = pool[1] if len(pool) > 1 else best

    k = 2
    while second.logic_id == best.logic_id and k < len(pool):
        second = pool[k]
        k += 1

    if best.logic_id == "096" and second.logic_id == "096":
        second = Candidate("001", {}, best.score + 0.1,
                           {"family": "Baseline", "name": "Identity", "valid": True})

    return best, second

# ------------------------------------------------------------------------------
# Apply candidate to test input
# ------------------------------------------------------------------------------
def apply_candidate_to_input(candidate: Candidate, Gin: np.ndarray) -> np.ndarray:
    if candidate.logic_id == "CHAIN2":
        chain = candidate.meta.get("chain", [])
        cur = Gin
        if chain and chain[0][0] == "000":
            c1 = Candidate(chain[1][0], chain[1][1], 0.0, {})
            pred = _apply_atomic_chain(chain[0][1], c1, Gin)
        else:
            for lid, params in chain:
                cur = _apply_step(lid, params, cur)
            pred = cur
    else:
        pred = _apply_step(candidate.logic_id, candidate.params, Gin)

    if candidate.logic_id != "096":
        bg = A02_background_mode(pred)
        pred = LOGIC_096_denoise(pred, bg=bg, min_same_neighbors=2)

    return A00_cast_int(pred)

print("✅ CELL 04 LOADED — Solver ready (single-step + CHAIN2 + Atomic Split).")


✅ CELL 04 — Candidate and Pair types ready.
✅ CELL 04 LOADED — Solver ready (single-step + CHAIN2 + Atomic Split).


In [8]:
# ==============================================================================
# CELL 04a.guard
# Solver Preflight Dependency Guard (Verbose, Final)
# ==============================================================================

print("\n" + "=" * 96)
print("[04a.guard] SOLVER PREFLIGHT DEPENDENCY CHECK")
print("=" * 96)

required = {
    "ARC": "CELL 01d (ARC dataset loader)",
    "parse_task_pairs": "CELL 01d (ARC dataset loader)",
    "REGISTRY": "CELL 03 (logic registry)",
    "hydrated_logic_ids": "CELL 03 (logic registry)",
    "induce_program": "CELL 04 (solver)",
    "apply_candidate_to_input": "CELL 04 (solver)",
}

missing = []

for sym, source in required.items():
    if sym not in globals():
        missing.append((sym, source))

if missing:
    print("[04a.guard] ❌ Missing required symbols:\n")
    for sym, source in missing:
        print(f"  - {sym:<25s} (defined in {source})")

    print("\n[04a.guard] ❌ Solver preflight BLOCKED")
    print("[04a.guard] Run the missing cells IN ORDER, then re-run CELL 04a.\n")

    raise RuntimeError(
        "Solver preflight blocked due to missing dependencies.\n"
        "See printed list above."
    )

print("[04a.guard] ✅ All solver dependencies present")
print("[04a.guard] ✅ Safe to run CELL 04a")


[04a.guard] SOLVER PREFLIGHT DEPENDENCY CHECK
[04a.guard] ✅ All solver dependencies present
[04a.guard] ✅ Safe to run CELL 04a


In [9]:
# ==============================================================================
# CELL 05
# Submission Generation (Canonical, ARC-AGI-2, Offline)
# Writes: /kaggle/working/submission.json
# ==============================================================================

import json
import numpy as np
from pathlib import Path

print("\n" + "=" * 96)
print("[05] SUBMISSION GENERATION — ARC-AGI-2")
print("=" * 96)

OUT_PATH = Path("/kaggle/working/submission.json")

# ------------------------------------------------------------------------------
# Safety checks
# ------------------------------------------------------------------------------
required = [
    "ARC",
    "induce_program",
    "apply_candidate_to_input",
    "parse_task_pairs",
]

missing = [n for n in required if n not in globals()]
if missing:
    raise RuntimeError(
        "Submission generation cannot proceed. "
        f"Missing required symbols: {missing}"
    )

# ------------------------------------------------------------------------------
# Generate submission
# ------------------------------------------------------------------------------
submission = {}

task_ids = sorted(ARC.eval_challenges.keys())
print(f"[05] Generating predictions for {len(task_ids)} evaluation tasks")

for tid in task_ids:
    task = ARC.eval_challenges[tid]
    train_pairs, test_inputs = parse_task_pairs(task)

    # Induce program from training examples
    cand1, cand2 = induce_program(train_pairs)

    preds = []
    for Gin in test_inputs:
        out1 = apply_candidate_to_input(cand1, Gin)
        out2 = apply_candidate_to_input(cand2, Gin)

        preds.append({
            "attempt_1": out1.astype(int).tolist(),
            "attempt_2": out2.astype(int).tolist(),
        })

    submission[tid] = preds

# ------------------------------------------------------------------------------
# Write submission.json
# ------------------------------------------------------------------------------
with open(OUT_PATH, "w") as f:
    json.dump(submission, f, indent=2)

print(f"[05] ✅ Submission written to {OUT_PATH}")
print(f"[05] Total tasks: {len(submission)}")


[05] SUBMISSION GENERATION — ARC-AGI-2
[05] Generating predictions for 120 evaluation tasks
[05] ✅ Submission written to /kaggle/working/submission.json
[05] Total tasks: 120


In [10]:
def diag_cell_05():
    return {
        "tasks_processed": len(EXECUTION_LOG),
        "avg_time": float(np.mean([e["data"]["time"] for e in EXECUTION_LOG])) if EXECUTION_LOG else 0,
        "max_time": float(np.max([e["data"]["time"] for e in EXECUTION_LOG])) if EXECUTION_LOG else 0,
        "sample_log": EXECUTION_LOG[:2],
        "metrics": {
            "scoring": "mean(pixel match)",
            "selection": "top_k ranking",
            "fallback": "identity fill"
        }
    }

print(diag_cell_05())

{'tasks_processed': 0, 'avg_time': 0, 'max_time': 0, 'sample_log': [], 'metrics': {'scoring': 'mean(pixel match)', 'selection': 'top_k ranking', 'fallback': 'identity fill'}}


In [11]:
# CELL 05b
# Submission Shape Normalization (Prevents Kaggle Scoring Error)

def normalize_submission_shapes(submission: dict) -> dict:
    """
    Ensures attempt_1 and attempt_2 have identical shapes per task entry.
    If a mismatch is found, attempt_2 is replaced with attempt_1.
    """
    fixes = 0

    for task_id, attempts in submission.items():
        for entry in attempts:
            a1 = entry.get("attempt_1")
            a2 = entry.get("attempt_2")

            if not a1 or not a2:
                continue

            h1, w1 = len(a1), len(a1[0])
            h2, w2 = len(a2), len(a2[0])

            if (h1 != h2) or (w1 != w2):
                entry["attempt_2"] = a1
                fixes += 1

    print(f"[05b] Fixed attempt shape mismatches in {fixes} entries")
    return submission

# Apply normalization BEFORE writing submission.json
submission = normalize_submission_shapes(submission)

[05b] Fixed attempt shape mismatches in 10 entries


In [12]:
# CELL 05e
# Guard against empty or degenerate outputs

def is_valid_grid(grid):
    if not grid:
        return False
    if not all(isinstance(row, list) and row for row in grid):
        return False
    return True

empty_fixes = 0

for task_id, entries in submission.items():
    for entry in entries:
        if not is_valid_grid(entry["attempt_1"]):
            entry["attempt_1"] = [[0]]
            empty_fixes += 1

        if not is_valid_grid(entry["attempt_2"]):
            entry["attempt_2"] = entry["attempt_1"]
            empty_fixes += 1

print(f"[05e] Repaired empty/invalid outputs in {empty_fixes} attempts")

[05e] Repaired empty/invalid outputs in 0 attempts


In [13]:
# ==============================================================================
# CELL 05f
# Submission Arbitration Boundary (Canonical, Deterministic)
#
# Defines the contract between:
#   - raw solver output
#   - post-solver arbitration
#   - final submission
# ==============================================================================

import copy

print("\n" + "=" * 96)
print("[05f] SUBMISSION ARBITRATION BOUNDARY")
print("=" * 96)

# ----------------------------------------------------------------------
# Hard requirement: submission must exist from CELL 05
# ----------------------------------------------------------------------
if "submission" not in globals():
    raise RuntimeError(
        "CELL 05f requires `submission` to be defined.\n"
        "Ensure CELL 05 (Submission Generation) has run."
    )

# ----------------------------------------------------------------------
# Freeze raw solver output
# ----------------------------------------------------------------------
raw_submission = copy.deepcopy(submission)

# This object will be mutated by arbitration stages ONLY
final_submission = copy.deepcopy(raw_submission)

print("[05f] ✅ Raw solver output frozen as `raw_submission`")
print("[05f] ✅ Arbitration target initialized as `final_submission`")


[05f] SUBMISSION ARBITRATION BOUNDARY
[05f] ✅ Raw solver output frozen as `raw_submission`
[05f] ✅ Arbitration target initialized as `final_submission`


In [14]:
# ==============================================================================
# CELL 05g
# Master Arbitration Configuration (Authoritative, Deterministic)
#
# This cell is the SINGLE source of truth for all post-solver arbitration policy.
# No arbitration stage may introduce new parameters outside this config.
#
# Safe to edit between runs.
# ==============================================================================
print("\n" + "=" * 96)
print("[05g] MASTER ARBITRATION CONFIGURATION")
print("=" * 96)

# ------------------------------------------------------------------------------
# Global arbitration enable switch
# ------------------------------------------------------------------------------
ENABLE_ARBITRATION = True

# ------------------------------------------------------------------------------
# Stage enable flags
# ------------------------------------------------------------------------------
ENABLE_STAGE_07 = True   # Bidirectional semantic stability
ENABLE_STAGE_08 = True   # Background dominance prior

# ------------------------------------------------------------------------------
# Stage 07 — Semantic Stability Parameters
# ------------------------------------------------------------------------------
SEMANTIC_COLOR_PENALTY_WEIGHT = 1.5
SEMANTIC_BACKGROUND_PENALTY_WEIGHT = 5.0
SEMANTIC_HISTOGRAM_PENALTY_WEIGHT = 0.01

# ------------------------------------------------------------------------------
# Stage 08 — Background Dominance Parameters
# ------------------------------------------------------------------------------
BACKGROUND_DOMINANT_THRESHOLD = 0.60
BACKGROUND_FREE_THRESHOLD = 0.05
MIN_BACKGROUND_DELTA = 0.25

# ------------------------------------------------------------------------------
# Arbitration invariants (documentary, not enforced by code)
# ------------------------------------------------------------------------------
ARBITRATION_INVARIANTS = {
    "may_generate_new_grids": False,
    "may_swap_attempt_order": True,
    "may_modify_attempt_contents": False,
    "deterministic_only": True,
    "offline_only": True,
}

# ------------------------------------------------------------------------------
# Config summary (human-readable)
# ------------------------------------------------------------------------------
print("[05g] Arbitration enabled           :", ENABLE_ARBITRATION)
print("[05g] Stage 07 (semantic stability) :", ENABLE_STAGE_07)
print("[05g] Stage 08 (background prior)   :", ENABLE_STAGE_08)
print("[05g] Invariants                    :", ARBITRATION_INVARIANTS)

print("[05g] ✅ Master arbitration config loaded")


[05g] MASTER ARBITRATION CONFIGURATION
[05g] Arbitration enabled           : True
[05g] Stage 07 (semantic stability) : True
[05g] Stage 08 (background prior)   : True
[05g] Invariants                    : {'may_generate_new_grids': False, 'may_swap_attempt_order': True, 'may_modify_attempt_contents': False, 'deterministic_only': True, 'offline_only': True}
[05g] ✅ Master arbitration config loaded


In [15]:
# ==============================================================================
# CELL 06
# Arbitration Utilities & Structural Feature Extraction (Read-Only)
# ==============================================================================

from collections import Counter

def grid_shape(grid):
    if not grid:
        return (0, 0)
    return (len(grid), len(grid[0]))

def is_rectangular(grid):
    if not grid:
        return False
    w = len(grid[0])
    return all(len(row) == w for row in grid)

def color_histogram(grid):
    counter = Counter()
    for row in grid:
        for v in row:
            counter[int(v)] += 1
    return dict(counter)

def grid_stats(grid):
    h, w = grid_shape(grid)
    hist = color_histogram(grid)
    return {
        "height": h,
        "width": w,
        "area": h * w,
        "unique_colors": len(hist),
        "background_ratio": hist.get(0, 0) / max(1, h * w),
        "histogram": hist,
        "rectangular": is_rectangular(grid),
    }

def grids_equal(g1, g2):
    return g1 == g2

print("✅ CELL 06 LOADED — Arbitration utilities ready")

✅ CELL 06 LOADED — Arbitration utilities ready


In [16]:
# ==============================================================================
# CELL 06a
# Arbitration Dataset Derivation (from final_submission)
# ==============================================================================

dataset_attempt_pairs = []
solver_behavior = {}

for task_id, entries in final_submission.items():
    solver_behavior[task_id] = {
        "task_id": task_id,
        "num_entries": len(entries),
        "disagreement_count": 0,
        "geometry_mismatch_count": 0,
        "identical_output_count": 0,
        "arbitration_swaps": 0,
    }

    for test_index, entry in enumerate(entries):
        a1 = entry["attempt_1"]
        a2 = entry["attempt_2"]

        stats1 = grid_stats(a1)
        stats2 = grid_stats(a2)

        identical = grids_equal(a1, a2)
        same_shape = (
            stats1["height"],
            stats1["width"],
        ) == (
            stats2["height"],
            stats2["width"],
        )

        if not identical:
            solver_behavior[task_id]["disagreement_count"] += 1
        if not same_shape:
            solver_behavior[task_id]["geometry_mismatch_count"] += 1
        if identical:
            solver_behavior[task_id]["identical_output_count"] += 1

        dataset_attempt_pairs.append({
            "task_id": task_id,
            "test_index": test_index,
            "attempt_1": stats1,
            "attempt_2": stats2,
            "same_shape": same_shape,
            "identical_output": identical,
        })

print(f"[06a] ✅ Derived {len(dataset_attempt_pairs)} arbitration records")

[06a] ✅ Derived 172 arbitration records


In [17]:
# ==============================================================================
# CELL 07
# Arbitration Stage 1 — Bidirectional Semantic Stability
# ==============================================================================

from math import fabs

ENABLE_STAGE_07 = True

def histogram_distance(h1, h2):
    keys = set(h1) | set(h2)
    return sum(abs(h1.get(k, 0) - h2.get(k, 0)) for k in keys)

def semantic_penalty(stats_ref, stats_cmp):
    penalty = 0.0
    penalty += max(0, stats_cmp["unique_colors"] - stats_ref["unique_colors"]) * 1.5
    penalty += fabs(stats_cmp["background_ratio"] - stats_ref["background_ratio"]) * 5.0
    penalty += histogram_distance(
        stats_ref["histogram"],
        stats_cmp["histogram"]
    ) * 0.01
    return penalty

reranked_stage_07 = 0

if ENABLE_STAGE_07:
    for record in dataset_attempt_pairs:
        if record["identical_output"]:
            continue

        stats1 = record["attempt_1"]
        stats2 = record["attempt_2"]

        p1 = semantic_penalty(stats1, stats2)
        p2 = semantic_penalty(stats2, stats1)

        if p2 < p1:
            task_id = record["task_id"]
            idx = record["test_index"]

            entry = final_submission[task_id][idx]
            entry["attempt_1"], entry["attempt_2"] = (
                entry["attempt_2"],
                entry["attempt_1"],
            )

            solver_behavior[task_id]["arbitration_swaps"] += 1
            reranked_stage_07 += 1

print(f"[07] ✅ Stage 1 applied — swaps: {reranked_stage_07}")

[07] ✅ Stage 1 applied — swaps: 22


In [18]:
# ==============================================================================
# CELL 08
# Arbitration Stage 2 — Background Dominance Prior
# ==============================================================================

ENABLE_STAGE_08 = True

BACKGROUND_DOMINANT_THRESHOLD = 0.6
BACKGROUND_FREE_THRESHOLD = 0.05
MIN_BACKGROUND_DELTA = 0.25

reranked_stage_08 = 0

if ENABLE_STAGE_08:
    for record in dataset_attempt_pairs:
        if record["identical_output"]:
            continue

        stats1 = record["attempt_1"]
        stats2 = record["attempt_2"]

        bg1 = stats1["background_ratio"]
        bg2 = stats2["background_ratio"]

        if abs(bg1 - bg2) < MIN_BACKGROUND_DELTA:
            continue

        prefer_1 = False
        prefer_2 = False

        if bg1 >= BACKGROUND_DOMINANT_THRESHOLD and bg2 < BACKGROUND_DOMINANT_THRESHOLD:
            prefer_1 = True
        elif bg2 >= BACKGROUND_DOMINANT_THRESHOLD and bg1 < BACKGROUND_DOMINANT_THRESHOLD:
            prefer_2 = True

        if bg1 <= BACKGROUND_FREE_THRESHOLD and bg2 > BACKGROUND_FREE_THRESHOLD:
            prefer_1 = True
        elif bg2 <= BACKGROUND_FREE_THRESHOLD and bg1 > BACKGROUND_FREE_THRESHOLD:
            prefer_2 = True

        if not (prefer_1 ^ prefer_2):
            continue

        task_id = record["task_id"]
        idx = record["test_index"]
        entry = final_submission[task_id][idx]

        if prefer_2:
            entry["attempt_1"], entry["attempt_2"] = (
                entry["attempt_2"],
                entry["attempt_1"],
            )
            solver_behavior[task_id]["arbitration_swaps"] += 1
            reranked_stage_08 += 1

print(f"[08] ✅ Stage 2 applied — swaps: {reranked_stage_08}")

[08] ✅ Stage 2 applied — swaps: 3


In [19]:
# ==============================================================================
# CELL 09
# Deep Diagnostics & Signal Extraction (Read-Only, Canonical-Safe)
#
# This cell extracts high-resolution behavioral signal from the solver output
# WITHOUT modifying any predictions or arbitration decisions.
# ==============================================================================
from collections import Counter
import math
import json

print("\n" + "=" * 96)
print("[09] DEEP DIAGNOSTICS — SIGNAL EXTRACTION")
print("=" * 96)

# ----------------------------------------------------------------------
# Safety checks
# ----------------------------------------------------------------------
required = [
    "final_submission",
    "solver_behavior",
]
missing = [n for n in required if n not in globals()]
if missing:
    raise RuntimeError(f"[09] Missing required symbols: {missing}")

# ----------------------------------------------------------------------
# Helpers
# ----------------------------------------------------------------------
def entropy_from_hist(hist):
    total = sum(hist.values())
    if total == 0:
        return 0.0
    H = 0.0
    for c in hist.values():
        p = c / total
        H -= p * math.log2(p)
    return H

def grid_entropy(grid):
    hist = Counter()
    for row in grid:
        for v in row:
            hist[int(v)] += 1
    return entropy_from_hist(hist), hist

def disagreement_ratio(g1, g2):
    if not g1 or not g2:
        return 1.0
    h1, w1 = len(g1), len(g1[0])
    h2, w2 = len(g2), len(g2[0])
    if (h1, w1) != (h2, w2):
        return 1.0

    mism = 0
    total = h1 * w1
    for y in range(h1):
        for x in range(w1):
            if g1[y][x] != g2[y][x]:
                mism += 1

    return mism / max(1, total)

def is_uniform(grid):
    if not grid:
        return True
    v0 = grid[0][0]
    return all(v == v0 for row in grid for v in row)

# ----------------------------------------------------------------------
# Accumulators
# ----------------------------------------------------------------------
global_stats = Counter()
confidence_buckets = Counter()
per_task_diagnostics = {}

# ----------------------------------------------------------------------
# Main analysis loop
# ----------------------------------------------------------------------
for task_id, entries in final_submission.items():
    task_diag = {
        "test_cases": [],
        "summary": Counter(),
    }

    swaps = solver_behavior.get(task_id, {}).get("arbitration_swaps", 0)

    for entry in entries:
        g1 = entry["attempt_1"]
        g2 = entry["attempt_2"]

        same = (g1 == g2)
        same_shape = (len(g1), len(g1[0])) == (len(g2), len(g2[0]))

        ent1, hist1 = grid_entropy(g1)
        ent2, hist2 = grid_entropy(g2)

        d_ratio = disagreement_ratio(g1, g2)
        bg1 = hist1.get(0, 0) / max(1, sum(hist1.values()))
        bg2 = hist2.get(0, 0) / max(1, sum(hist2.values()))

        uniform1 = is_uniform(g1)
        uniform2 = is_uniform(g2)

        task_diag["test_cases"].append({
            "same_output": same,
            "same_shape": same_shape,
            "disagreement_ratio": round(d_ratio, 4),
            "entropy_1": round(ent1, 4),
            "entropy_2": round(ent2, 4),
            "entropy_delta": round(ent1 - ent2, 4),
            "background_ratio_1": round(bg1, 4),
            "background_ratio_2": round(bg2, 4),
            "uniform_1": uniform1,
            "uniform_2": uniform2,
        })

        if uniform1 and uniform2:
            global_stats["uniform_collapse"] += 1
        if not same_shape:
            global_stats["shape_mismatch"] += 1
        if same:
            global_stats["identical_attempts"] += 1

        if same and ent1 > 1.0:
            confidence_buckets["high"] += 1
        elif d_ratio < 0.05:
            confidence_buckets["medium"] += 1
        else:
            confidence_buckets["low"] += 1

    task_diag["summary"]["arbitration_swaps"] = swaps
    per_task_diagnostics[task_id] = task_diag

# ----------------------------------------------------------------------
# Global summary
# ----------------------------------------------------------------------
summary = {
    "total_tasks": len(final_submission),
    "total_test_cases": sum(len(v) for v in final_submission.values()),
    "global_stats": dict(global_stats),
    "confidence_buckets": dict(confidence_buckets),
}

print("[09] SUMMARY")
print("-" * 96)
print(json.dumps(summary, indent=2))
print("\n[09] ✅ Deep diagnostics complete")


[09] DEEP DIAGNOSTICS — SIGNAL EXTRACTION
[09] SUMMARY
------------------------------------------------------------------------------------------------
{
  "total_tasks": 120,
  "total_test_cases": 172,
  "global_stats": {
    "identical_attempts": 124,
    "uniform_collapse": 19
  },
  "confidence_buckets": {
    "high": 51,
    "medium": 78,
    "low": 43
  }
}

[09] ✅ Deep diagnostics complete


In [20]:
# CELL 10
# Per-Task Confidence Scoring (Read-Only, Canonical-Safe)
# ==============================================================================

print("\n" + "=" * 96)
print("[10] PER-TASK CONFIDENCE SCORING")
print("=" * 96)

task_confidence = {}

for task_id, diag in per_task_diagnostics.items():
    cases = diag["test_cases"]
    if not cases:
        task_confidence[task_id] = 0.0
        continue

    score = 0.0
    max_score = 0.0

    for c in cases:
        max_score += 1.0
        s = 0.0

        if c["same_output"]:
            s += 0.4
        if c["disagreement_ratio"] < 0.05:
            s += 0.3
        if c["entropy_1"] > 1.0:
            s += 0.2
        if c["uniform_1"] and c["uniform_2"]:
            s -= 0.6

        score += max(0.0, s)

    task_confidence[task_id] = round(score / max(1.0, max_score), 4)

values = list(task_confidence.values())
print("[10] TASK CONFIDENCE SUMMARY")
print(f"  Tasks evaluated : {len(values)}")
print(f"  Mean confidence : {round(sum(values) / max(1, len(values)), 4)}")
print(f"  Min confidence  : {round(min(values), 4)}")
print(f"  Max confidence  : {round(max(values), 4)}")

print("\n[10] ✅ Per-task confidence scores computed")


[10] PER-TASK CONFIDENCE SCORING
[10] TASK CONFIDENCE SUMMARY
  Tasks evaluated : 120
  Mean confidence : 0.5683
  Min confidence  : 0.0
  Max confidence  : 0.9

[10] ✅ Per-task confidence scores computed


In [21]:
# CELL 11
# Entropy Delta vs Arbitration Swaps Correlation
# ==============================================================================

print("\n" + "=" * 96)
print("[11] ENTROPY Δ vs ARBITRATION ANALYSIS")
print("=" * 96)

pairs = []

for task_id, diag in per_task_diagnostics.items():
    swaps = diag["summary"].get("arbitration_swaps", 0)
    deltas = [abs(c["entropy_delta"]) for c in diag["test_cases"]]
    if deltas:
        avg_delta = sum(deltas) / len(deltas)
        pairs.append((avg_delta, swaps))

if not pairs:
    print("[11] No data available for correlation")
else:
    xs = [p[0] for p in pairs]
    ys = [p[1] for p in pairs]
    n = len(xs)

    mean_x = sum(xs) / n
    mean_y = sum(ys) / n

    num = sum((x - mean_x) * (y - mean_y) for x, y in zip(xs, ys))
    den_x = math.sqrt(sum((x - mean_x) ** 2 for x in xs))
    den_y = math.sqrt(sum((y - mean_y) ** 2 for y in ys))

    corr = 0.0 if den_x == 0 or den_y == 0 else num / (den_x * den_y)

    print(f"[11] Tasks analyzed          : {n}")
    print(f"[11] Avg |entropy delta|     : {round(mean_x, 4)}")
    print(f"[11] Avg arbitration swaps   : {round(mean_y, 4)}")
    print(f"[11] Pearson correlation (r) : {round(corr, 4)}")

print("\n[11] ✅ Entropy/arbitration analysis complete")


[11] ENTROPY Δ vs ARBITRATION ANALYSIS
[11] Tasks analyzed          : 120
[11] Avg |entropy delta|     : 0.0979
[11] Avg arbitration swaps   : 0.2083
[11] Pearson correlation (r) : 0.6164

[11] ✅ Entropy/arbitration analysis complete


In [22]:
# CELL 12
# Diagnostics JSON Export (Offline, Deterministic)
# ==============================================================================

print("\n" + "=" * 96)
print("[12] EXPORTING DIAGNOSTICS JSON")
print("=" * 96)

export_payload = {
    "summary": summary,
    "global_stats": summary.get("global_stats", {}),
    "confidence_buckets": summary.get("confidence_buckets", {}),
    "per_task_confidence": task_confidence,
    "per_task_diagnostics": per_task_diagnostics,
}

output_path = "arc_diagnostics_v1.json"

with open(output_path, "w") as f:
    json.dump(export_payload, f, indent=2)

print(f"[12] Diagnostics written to: {output_path}")
print("[12] ✅ Export complete")


[12] EXPORTING DIAGNOSTICS JSON
[12] Diagnostics written to: arc_diagnostics_v1.json
[12] ✅ Export complete


In [23]:
# CELL 13
# Failure-Mode Clustering (Read-Only, Diagnostic Only)
#
# This cell groups ARC tasks into interpretable failure modes using
# existing diagnostics (entropy, disagreement, uniformity, arbitration).
# NO solver behavior is modified.
# ==============================================================================

from collections import Counter, defaultdict
import json

print("\n" + "=" * 96)
print("[13] FAILURE-MODE CLUSTERING")
print("=" * 96)

# ----------------------------------------------------------------------
# Cluster definitions (explicit, interpretable rules)
# ----------------------------------------------------------------------
clusters = defaultdict(list)
cluster_stats = Counter()

for task_id, diag in per_task_diagnostics.items():
    cases = diag["test_cases"]
    swaps = diag["summary"].get("arbitration_swaps", 0)
    conf = task_confidence.get(task_id, 0.0)

    # Aggregate signals per task
    any_uniform = any(c["uniform_1"] and c["uniform_2"] for c in cases)
    max_entropy_delta = max(abs(c["entropy_delta"]) for c in cases) if cases else 0.0
    max_disagreement = max(c["disagreement_ratio"] for c in cases) if cases else 0.0
    avg_background = sum(
        max(c["background_ratio_1"], c["background_ratio_2"]) for c in cases
    ) / max(1, len(cases))

    # ------------------------------------------------------------------
    # Failure mode assignment (ordered, first-match wins)
    # ------------------------------------------------------------------
    if any_uniform:
        mode = "uniform_collapse"

    elif max_entropy_delta > 0.75 and max_disagreement > 0.3:
        mode = "high_entropy_disagreement"

    elif avg_background > 0.85 and conf < 0.6:
        mode = "background_dominant"

    elif swaps >= 2 and conf < 0.7:
        mode = "arbitration_unstable"

    elif conf >= 0.7:
        mode = "high_confidence_stable"

    else:
        mode = "other_ambiguous"

    clusters[mode].append(task_id)
    cluster_stats[mode] += 1

# ----------------------------------------------------------------------
# Reporting
# ----------------------------------------------------------------------
total_tasks = len(per_task_diagnostics)

print("[13] FAILURE MODE DISTRIBUTION")
print("-" * 96)

for mode, count in cluster_stats.most_common():
    pct = round(100 * count / max(1, total_tasks), 2)
    print(f"{mode:25s} : {count:3d} tasks ({pct:5.2f}%)")

# ----------------------------------------------------------------------
# Detailed cluster export (optional downstream use)
# ----------------------------------------------------------------------
failure_mode_report = {
    "total_tasks": total_tasks,
    "cluster_distribution": dict(cluster_stats),
    "clusters": dict(clusters),
}

print("\n[13] CLUSTER SUMMARY (JSON PREVIEW)")
print("-" * 96)
print(json.dumps(
    {k: len(v) for k, v in clusters.items()},
    indent=2
))

print("\n[13] ✅ Failure-mode clustering complete")


[13] FAILURE-MODE CLUSTERING
[13] FAILURE MODE DISTRIBUTION
------------------------------------------------------------------------------------------------
high_confidence_stable    :  75 tasks (62.50%)
other_ambiguous           :  21 tasks (17.50%)
uniform_collapse          :  11 tasks ( 9.17%)
background_dominant       :   5 tasks ( 4.17%)
high_entropy_disagreement :   4 tasks ( 3.33%)
arbitration_unstable      :   4 tasks ( 3.33%)

[13] CLUSTER SUMMARY (JSON PREVIEW)
------------------------------------------------------------------------------------------------
{
  "high_confidence_stable": 75,
  "background_dominant": 5,
  "other_ambiguous": 21,
  "high_entropy_disagreement": 4,
  "uniform_collapse": 11,
  "arbitration_unstable": 4
}

[13] ✅ Failure-mode clustering complete


In [24]:
# CELL 14
# Scoped Entropy-Aware Arbitration Signal (Read-Only, Canonical-Safe)
#
# This cell computes a RECOMMENDED arbitration preference using
# balanced-entropy logic, scoped strictly to failure modes where
# entropy disagreement is meaningful.
#
# NO solver outputs are modified.
# NO arbitration decisions are enforced.
# ==============================================================================

from collections import defaultdict
import math
import json

print("\n" + "=" * 96)
print("[14] SCOPED ENTROPY-AWARE ARBITRATION SIGNAL")
print("=" * 96)

# ----------------------------------------------------------------------
# Scope definition (derived from CELL 13 clustering)
# ----------------------------------------------------------------------
ENTROPY_ELIGIBLE_MODES = {
    "high_entropy_disagreement",
    "other_ambiguous",
}

# We reconstruct task -> cluster lookup from CELL 13 results
task_to_cluster = {}
for mode, task_ids in clusters.items():
    for tid in task_ids:
        task_to_cluster[tid] = mode

# ----------------------------------------------------------------------
# Entropy preference computation
# ----------------------------------------------------------------------
entropy_recommendations = {}
stats = defaultdict(int)

for task_id, diag in per_task_diagnostics.items():
    cluster = task_to_cluster.get(task_id, "unknown")
    cases = diag["test_cases"]

    # Default: no recommendation
    recommendation = {
        "eligible": False,
        "preferred_attempt": None,
        "reason": "not_entropy_eligible",
    }

    # Scope guard
    if cluster not in ENTROPY_ELIGIBLE_MODES:
        entropy_recommendations[task_id] = recommendation
        stats["skipped_out_of_scope"] += 1
        continue

    # Aggregate per-task entropy signals
    deltas = []
    ent1s = []
    ent2s = []

    for c in cases:
        ent1s.append(c["entropy_1"])
        ent2s.append(c["entropy_2"])
        deltas.append(abs(c["entropy_delta"]))

    if not deltas:
        entropy_recommendations[task_id] = recommendation
        stats["skipped_no_data"] += 1
        continue

    avg_ent1 = sum(ent1s) / len(ent1s)
    avg_ent2 = sum(ent2s) / len(ent2s)
    avg_delta = sum(deltas) / len(deltas)

    # ------------------------------------------------------------------
    # Balanced entropy rule:
    # Prefer the attempt whose entropy is closer to the midpoint
    # (discourages collapse AND discourages over-noise)
    # ------------------------------------------------------------------
    midpoint = (avg_ent1 + avg_ent2) / 2.0
    dist1 = abs(avg_ent1 - midpoint)
    dist2 = abs(avg_ent2 - midpoint)

    if dist1 < dist2:
        preferred = "attempt_1"
        reason = "balanced_entropy_preference"
    elif dist2 < dist1:
        preferred = "attempt_2"
        reason = "balanced_entropy_preference"
    else:
        preferred = None
        reason = "entropy_tie"

    recommendation = {
        "eligible": True,
        "cluster": cluster,
        "avg_entropy_1": round(avg_ent1, 4),
        "avg_entropy_2": round(avg_ent2, 4),
        "avg_entropy_delta": round(avg_delta, 4),
        "preferred_attempt": preferred,
        "reason": reason,
    }

    entropy_recommendations[task_id] = recommendation
    stats["entropy_evaluated"] += 1
    if preferred is not None:
        stats["preference_emitted"] += 1

# ----------------------------------------------------------------------
# Reporting
# ----------------------------------------------------------------------
print("[14] ENTROPY SIGNAL SUMMARY")
print("-" * 96)

for k, v in stats.items():
    print(f"{k:30s} : {v}")

# Preview a small sample (deterministic order)
print("\n[14] SAMPLE RECOMMENDATIONS")
print("-" * 96)

sample = list(entropy_recommendations.items())[:5]
print(json.dumps(dict(sample), indent=2))

print("\n[14] ✅ Scoped entropy-aware arbitration signal computed")


[14] SCOPED ENTROPY-AWARE ARBITRATION SIGNAL
[14] ENTROPY SIGNAL SUMMARY
------------------------------------------------------------------------------------------------
skipped_out_of_scope           : 95
entropy_evaluated              : 25
preference_emitted             : 7

[14] SAMPLE RECOMMENDATIONS
------------------------------------------------------------------------------------------------
{
  "0934a4d8": {
    "eligible": false,
    "preferred_attempt": null,
    "reason": "not_entropy_eligible"
  },
  "135a2760": {
    "eligible": false,
    "preferred_attempt": null,
    "reason": "not_entropy_eligible"
  },
  "136b0064": {
    "eligible": false,
    "preferred_attempt": null,
    "reason": "not_entropy_eligible"
  },
  "13e47133": {
    "eligible": false,
    "preferred_attempt": null,
    "reason": "not_entropy_eligible"
  },
  "142ca369": {
    "eligible": false,
    "preferred_attempt": null,
    "reason": "not_entropy_eligible"
  }
}

[14] ✅ Scoped entropy-aware arbi

In [25]:
# CELL 14
# Scoped Entropy-Aware Arbitration Signal (Read-Only, Canonical-Safe)
#
# This cell computes a RECOMMENDED arbitration preference using
# balanced-entropy logic, scoped strictly to failure modes where
# entropy disagreement is meaningful.
#
# NO solver outputs are modified.
# NO arbitration decisions are enforced.
# ==============================================================================

from collections import defaultdict
import math
import json

print("\n" + "=" * 96)
print("[14] SCOPED ENTROPY-AWARE ARBITRATION SIGNAL")
print("=" * 96)

# ----------------------------------------------------------------------
# Scope definition (derived from CELL 13 clustering)
# ----------------------------------------------------------------------
ENTROPY_ELIGIBLE_MODES = {
    "high_entropy_disagreement",
    "other_ambiguous",
}

# We reconstruct task -> cluster lookup from CELL 13 results
task_to_cluster = {}
for mode, task_ids in clusters.items():
    for tid in task_ids:
        task_to_cluster[tid] = mode

# ----------------------------------------------------------------------
# Entropy preference computation
# ----------------------------------------------------------------------
entropy_recommendations = {}
stats = defaultdict(int)

for task_id, diag in per_task_diagnostics.items():
    cluster = task_to_cluster.get(task_id, "unknown")
    cases = diag["test_cases"]

    # Default: no recommendation
    recommendation = {
        "eligible": False,
        "preferred_attempt": None,
        "reason": "not_entropy_eligible",
    }

    # Scope guard
    if cluster not in ENTROPY_ELIGIBLE_MODES:
        entropy_recommendations[task_id] = recommendation
        stats["skipped_out_of_scope"] += 1
        continue

    # Aggregate per-task entropy signals
    deltas = []
    ent1s = []
    ent2s = []

    for c in cases:
        ent1s.append(c["entropy_1"])
        ent2s.append(c["entropy_2"])
        deltas.append(abs(c["entropy_delta"]))

    if not deltas:
        entropy_recommendations[task_id] = recommendation
        stats["skipped_no_data"] += 1
        continue

    avg_ent1 = sum(ent1s) / len(ent1s)
    avg_ent2 = sum(ent2s) / len(ent2s)
    avg_delta = sum(deltas) / len(deltas)

    # ------------------------------------------------------------------
    # Balanced entropy rule:
    # Prefer the attempt whose entropy is closer to the midpoint
    # (discourages collapse AND discourages over-noise)
    # ------------------------------------------------------------------
    midpoint = (avg_ent1 + avg_ent2) / 2.0
    dist1 = abs(avg_ent1 - midpoint)
    dist2 = abs(avg_ent2 - midpoint)

    if dist1 < dist2:
        preferred = "attempt_1"
        reason = "balanced_entropy_preference"
    elif dist2 < dist1:
        preferred = "attempt_2"
        reason = "balanced_entropy_preference"
    else:
        preferred = None
        reason = "entropy_tie"

    recommendation = {
        "eligible": True,
        "cluster": cluster,
        "avg_entropy_1": round(avg_ent1, 4),
        "avg_entropy_2": round(avg_ent2, 4),
        "avg_entropy_delta": round(avg_delta, 4),
        "preferred_attempt": preferred,
        "reason": reason,
    }

    entropy_recommendations[task_id] = recommendation
    stats["entropy_evaluated"] += 1
    if preferred is not None:
        stats["preference_emitted"] += 1

# ----------------------------------------------------------------------
# Reporting
# ----------------------------------------------------------------------
print("[14] ENTROPY SIGNAL SUMMARY")
print("-" * 96)

for k, v in stats.items():
    print(f"{k:30s} : {v}")

# Preview a small sample (deterministic order)
print("\n[14] SAMPLE RECOMMENDATIONS")
print("-" * 96)

sample = list(entropy_recommendations.items())[:5]
print(json.dumps(dict(sample), indent=2))

print("\n[14] ✅ Scoped entropy-aware arbitration signal computed")


[14] SCOPED ENTROPY-AWARE ARBITRATION SIGNAL
[14] ENTROPY SIGNAL SUMMARY
------------------------------------------------------------------------------------------------
skipped_out_of_scope           : 95
entropy_evaluated              : 25
preference_emitted             : 7

[14] SAMPLE RECOMMENDATIONS
------------------------------------------------------------------------------------------------
{
  "0934a4d8": {
    "eligible": false,
    "preferred_attempt": null,
    "reason": "not_entropy_eligible"
  },
  "135a2760": {
    "eligible": false,
    "preferred_attempt": null,
    "reason": "not_entropy_eligible"
  },
  "136b0064": {
    "eligible": false,
    "preferred_attempt": null,
    "reason": "not_entropy_eligible"
  },
  "13e47133": {
    "eligible": false,
    "preferred_attempt": null,
    "reason": "not_entropy_eligible"
  },
  "142ca369": {
    "eligible": false,
    "preferred_attempt": null,
    "reason": "not_entropy_eligible"
  }
}

[14] ✅ Scoped entropy-aware arbi

In [26]:
# CELL 15
# Uniform-Collapse Arbitration Veto (Read-Only, Canonical-Safe)
#
# This cell detects tasks where BOTH attempts collapse to uniform grids
# and emits a hard veto signal indicating the result should NOT be trusted,
# regardless of agreement or entropy.
#
# NO solver outputs are modified.
# NO arbitration decisions are enforced here.
# ==============================================================================

from collections import defaultdict
import json

print("\n" + "=" * 96)
print("[15] UNIFORM-COLLAPSE ARBITRATION VETO")
print("=" * 96)

uniform_veto = {}
stats = defaultdict(int)

for task_id, diag in per_task_diagnostics.items():
    cases = diag["test_cases"]

    # A task is vetoed if ANY test case has BOTH attempts uniform
    veto = any(c["uniform_1"] and c["uniform_2"] for c in cases)

    if veto:
        uniform_veto[task_id] = {
            "veto": True,
            "reason": "both_attempts_uniform",
            "confidence_override": 0.0
        }
        stats["vetoed_tasks"] += 1
    else:
        uniform_veto[task_id] = {
            "veto": False,
            "reason": "not_uniform_collapse"
        }
        stats["allowed_tasks"] += 1

# ----------------------------------------------------------------------
# Reporting
# ----------------------------------------------------------------------
total = len(per_task_diagnostics)

print("[15] VETO SUMMARY")
print("-" * 96)
print(f"Total tasks evaluated : {total}")
print(f"Uniform-collapse veto : {stats['vetoed_tasks']}")
print(f"Tasks allowed         : {stats['allowed_tasks']}")

# Sanity check against clustering
print("\n[15] SANITY CHECK (EXPECTED ≈ uniform_collapse cluster)")
print("-" * 96)
print("Expected ≈ 11 based on CELL 13 clustering")

# Preview vetoed task IDs
print("\n[15] VETOED TASK IDS (PREVIEW)")
print("-" * 96)
print(list(tid for tid, v in uniform_veto.items() if v["veto"])[:10])

print("\n[15] ✅ Uniform-collapse arbitration veto signal computed")


[15] UNIFORM-COLLAPSE ARBITRATION VETO
[15] VETO SUMMARY
------------------------------------------------------------------------------------------------
Total tasks evaluated : 120
Uniform-collapse veto : 11
Tasks allowed         : 109

[15] SANITY CHECK (EXPECTED ≈ uniform_collapse cluster)
------------------------------------------------------------------------------------------------
Expected ≈ 11 based on CELL 13 clustering

[15] VETOED TASK IDS (PREVIEW)
------------------------------------------------------------------------------------------------
['271d71e2', '2c181942', '446ef5d2', '5961cc34', 'a25697e4', 'abc82100', 'b6f77b65', 'cbebaa4b', 'd35bdbdc', 'd59b0160']

[15] ✅ Uniform-collapse arbitration veto signal computed


In [27]:
# CELL 16
# Confidence-Weighted Arbitration Governance
#
# This cell synthesizes ALL prior diagnostic signals into a single
# per-task control policy that governs:
# - whether arbitration should be trusted
# - whether entropy preference may be applied
# - whether confidence must be overridden
#
# NO solver outputs are modified.
# NO arbitration decisions are enforced here.
# ==============================================================================

import json

print("\n" + "=" * 96)
print("[16] CONFIDENCE-WEIGHTED ARBITRATION GOVERNANCE")
print("=" * 96)

# ----------------------------------------------------------------------
# Governance thresholds (explicit, conservative)
# ----------------------------------------------------------------------
HIGH_CONFIDENCE_THRESHOLD = 0.7
LOW_CONFIDENCE_THRESHOLD = 0.3

governance_policy = {}

for task_id in per_task_diagnostics.keys():
    conf = task_confidence.get(task_id, 0.0)
    veto_info = uniform_veto.get(task_id, {"veto": False})
    entropy_info = entropy_recommendations.get(task_id, {})
    cluster = task_to_cluster.get(task_id, "unknown")

    policy = {
        "task_id": task_id,
        "cluster": cluster,
        "confidence": conf,

        # Core control flags
        "allow_arbitration": False,
        "allow_entropy_preference": False,
        "confidence_override": None,

        # Explanatory fields (for auditability)
        "reasons": [],
    }

    # ------------------------------------------------------------------
    # Rule 1: Uniform-collapse veto (absolute priority)
    # ------------------------------------------------------------------
    if veto_info.get("veto", False):
        policy["allow_arbitration"] = False
        policy["allow_entropy_preference"] = False
        policy["confidence_override"] = 0.0
        policy["reasons"].append("uniform_collapse_veto")

        governance_policy[task_id] = policy
        continue

    # ------------------------------------------------------------------
    # Rule 2: High-confidence protection
    # ------------------------------------------------------------------
    if conf >= HIGH_CONFIDENCE_THRESHOLD:
        policy["allow_arbitration"] = False
        policy["allow_entropy_preference"] = False
        policy["reasons"].append("high_confidence_protected")

        governance_policy[task_id] = policy
        continue

    # ------------------------------------------------------------------
    # Rule 3: Low-confidence permissive arbitration
    # ------------------------------------------------------------------
    if conf <= LOW_CONFIDENCE_THRESHOLD:
        policy["allow_arbitration"] = True
        policy["reasons"].append("low_confidence_allows_arbitration")

    # ------------------------------------------------------------------
    # Rule 4: Entropy-aware arbitration (scoped)
    # ------------------------------------------------------------------
    if (
        entropy_info.get("eligible", False)
        and entropy_info.get("preferred_attempt") is not None
        and cluster in {"high_entropy_disagreement", "other_ambiguous"}
    ):
        policy["allow_entropy_preference"] = True
        policy["reasons"].append("scoped_entropy_preference_allowed")

    # ------------------------------------------------------------------
    # Default: medium-confidence conservative behavior
    # ------------------------------------------------------------------
    if not policy["reasons"]:
        policy["reasons"].append("medium_confidence_conservative")

    governance_policy[task_id] = policy

# ----------------------------------------------------------------------
# Reporting
# ----------------------------------------------------------------------
summary = {
    "total_tasks": len(governance_policy),
    "high_confidence_protected": sum(
        1 for p in governance_policy.values()
        if "high_confidence_protected" in p["reasons"]
    ),
    "uniform_vetoed": sum(
        1 for p in governance_policy.values()
        if "uniform_collapse_veto" in p["reasons"]
    ),
    "arbitration_allowed": sum(
        1 for p in governance_policy.values()
        if p["allow_arbitration"]
    ),
    "entropy_preference_allowed": sum(
        1 for p in governance_policy.values()
        if p["allow_entropy_preference"]
    ),
}

print("[16] GOVERNANCE SUMMARY")
print("-" * 96)
print(json.dumps(summary, indent=2))

print("\n[16] ✅ Confidence-weighted governance policy computed")


[16] CONFIDENCE-WEIGHTED ARBITRATION GOVERNANCE
[16] GOVERNANCE SUMMARY
------------------------------------------------------------------------------------------------
{
  "total_tasks": 120,
  "high_confidence_protected": 75,
  "uniform_vetoed": 11,
  "arbitration_allowed": 30,
  "entropy_preference_allowed": 7
}

[16] ✅ Confidence-weighted governance policy computed


In [28]:
# CELL 17
# Export Governance Policy (Offline, Deterministic)
#
# Writes the final per-task arbitration governance policy to disk
# for inspection, auditing, and reproducibility.
# ==============================================================================

import json

print("\n" + "=" * 96)
print("[17] EXPORTING GOVERNANCE POLICY")
print("=" * 96)

governance_export = {
    "summary": {
        "total_tasks": len(governance_policy),
        "high_confidence_protected": sum(
            1 for p in governance_policy.values()
            if "high_confidence_protected" in p["reasons"]
        ),
        "uniform_vetoed": sum(
            1 for p in governance_policy.values()
            if "uniform_collapse_veto" in p["reasons"]
        ),
        "arbitration_allowed": sum(
            1 for p in governance_policy.values()
            if p["allow_arbitration"]
        ),
        "entropy_preference_allowed": sum(
            1 for p in governance_policy.values()
            if p["allow_entropy_preference"]
        ),
    },
    "governance_policy": governance_policy,
}

output_path = "arc_governance_policy_v1.json"

with open(output_path, "w") as f:
    json.dump(governance_export, f, indent=2)

print(f"[17] Governance policy written to: {output_path}")
print("[17] ✅ Export complete")


[17] EXPORTING GOVERNANCE POLICY
[17] Governance policy written to: arc_governance_policy_v1.json
[17] ✅ Export complete


In [29]:
# CELL 18
# Governance-Enforced Arbitration Wrapper
#
# Applies the governance policy to determine which attempt is selected
# for each task, WITHOUT modifying solver behavior or predictions.
#
# This is the FINAL arbitration authority.
# ==============================================================================

print("\n" + "=" * 96)
print("[18] APPLYING GOVERNANCE TO ARBITRATION")
print("=" * 96)

final_governed_submission = {}

for task_id, entries in final_submission.items():
    policy = governance_policy.get(task_id, {})
    veto = policy.get("confidence_override") == 0.0
    allow_entropy = policy.get("allow_entropy_preference", False)
    entropy_info = entropy_recommendations.get(task_id, {})

    governed_entries = []

    for entry in entries:
        # Default behavior: keep original arbitration
        chosen = entry.get("final", None)

        # If uniform-collapse veto: force attempt_1 (arbitrary but explicit)
        if veto:
            chosen = entry["attempt_1"]

        # If entropy preference allowed and present
        elif allow_entropy:
            pref = entropy_info.get("preferred_attempt")
            if pref == "attempt_1":
                chosen = entry["attempt_1"]
            elif pref == "attempt_2":
                chosen = entry["attempt_2"]

        # Fallback: if no final chosen yet, default to attempt_1
        if chosen is None:
            chosen = entry["attempt_1"]

        governed_entries.append(chosen)

    final_governed_submission[task_id] = governed_entries

print("[18] ✅ Governance-enforced arbitration complete")
print(f"[18] Tasks governed : {len(final_governed_submission)}")


[18] APPLYING GOVERNANCE TO ARBITRATION
[18] ✅ Governance-enforced arbitration complete
[18] Tasks governed : 120


In [30]:
# CELL 19
# Governed vs Unguided Submission Diff (CORRECTED)
#
# This version compares FINAL GRID OUTPUTS only.
# It fixes the structural mismatch bug that caused 100% false positives.
# ==============================================================================

from collections import defaultdict
import json

print("\n" + "=" * 96)
print("[19] GOVERNED vs UNGUIDED SUBMISSION DIFF (CORRECTED)")
print("=" * 96)

diff_report = []
stats = defaultdict(int)
cluster_stats = defaultdict(int)

for task_id, original_entries in final_submission.items():
    governed_entries = final_governed_submission.get(task_id)
    policy = governance_policy.get(task_id, {})
    cluster = policy.get("cluster", "unknown")

    if governed_entries is None:
        continue

    changed = False

    for idx, entry in enumerate(original_entries):
        # Extract unguided final grid
        if "final" in entry and entry["final"] is not None:
            original_grid = entry["final"]
        else:
            # fallback: attempt_1 (matches prior arbitration behavior)
            original_grid = entry["attempt_1"]

        governed_grid = governed_entries[idx]

        if original_grid != governed_grid:
            changed = True
            break

    if not changed:
        stats["unchanged"] += 1
        continue

    diff_entry = {
        "task_id": task_id,
        "cluster": cluster,
        "confidence": policy.get("confidence"),
        "reasons": policy.get("reasons"),
        "allow_arbitration": policy.get("allow_arbitration"),
        "allow_entropy_preference": policy.get("allow_entropy_preference"),
    }

    diff_report.append(diff_entry)
    stats["changed"] += 1
    cluster_stats[cluster] += 1

# ----------------------------------------------------------------------
# Reporting
# ----------------------------------------------------------------------
total = len(final_submission)

print("[19] DIFF SUMMARY")
print("-" * 96)
print(f"Total tasks evaluated : {total}")
print(f"Tasks unchanged       : {stats['unchanged']}")
print(f"Tasks changed         : {stats['changed']}")

if total > 0:
    pct = round(100 * stats["changed"] / total, 2)
    print(f"Change rate           : {pct}%")

print("\n[19] CHANGES BY CLUSTER")
print("-" * 96)
for cluster, count in sorted(cluster_stats.items(), key=lambda x: -x[1]):
    print(f"{cluster:25s} : {count}")

print("\n[19] DETAILED CHANGE LOG (JSON)")
print("-" * 96)
print(json.dumps(diff_report, indent=2))

print("\n[19] ✅ Corrected governed vs unguided diff complete")


[19] GOVERNED vs UNGUIDED SUBMISSION DIFF (CORRECTED)
[19] DIFF SUMMARY
------------------------------------------------------------------------------------------------
Total tasks evaluated : 120
Tasks unchanged       : 116
Tasks changed         : 4
Change rate           : 3.33%

[19] CHANGES BY CLUSTER
------------------------------------------------------------------------------------------------
other_ambiguous           : 3
high_entropy_disagreement : 1

[19] DETAILED CHANGE LOG (JSON)
------------------------------------------------------------------------------------------------
[
  {
    "task_id": "65b59efc",
    "cluster": "other_ambiguous",
    "confidence": 0.2,
    "reasons": [
      "low_confidence_allows_arbitration",
      "scoped_entropy_preference_allowed"
    ],
    "allow_arbitration": true,
    "allow_entropy_preference": true
  },
  {
    "task_id": "898e7135",
    "cluster": "high_entropy_disagreement",
    "confidence": 0.2,
    "reasons": [
      "low_confiden